# Phase 3 — Product Detection (YOLOv8 fine-tuned on SKU-110K)

Runs on either **Google Colab** or **Kaggle Notebooks** with a GPU (T4). **Kaggle is
recommended for this training run**: its free-tier sessions run 9-12 hours and support
background execution, vs. Colab's shorter session cap — that headroom matters for a
multi-hour fine-tune like this one.

## Why SKU-110K, not Freiburg

Freiburg Groceries (from Phase 2) is a *classification* dataset — one label per whole image,
no bounding boxes. YOLO needs bounding boxes to learn detection. SKU-110K is purpose-built for
this: 11,743 real supermarket shelf photos with 1.7M annotated bounding boxes.

It's **single-class** (every box is just "product," not "cereal" vs "milk") — that's actually
the right scope here. YOLO's job in this pipeline is to *localize* a product in the frame so it
can be cropped and handed to OCR/VQA; figuring out *what* the product specifically is happens
downstream, not inside the detector.

License note: SKU-110K is for academic/non-commercial use — fine for a portfolio project.

## Setup

In [ ]:
!pip install -q ultralytics wandb

from ultralytics import YOLO
import wandb

# Log in to W&B for experiment tracking (will prompt for your API key the first time —
# get one free at wandb.ai/authorize)
wandb.login()

## Dataset

Ultralytics ships a built-in config for SKU-110K, so we don't need to hand-write a download
script — pointing `model.train()` at `sku-110k.yaml` triggers an automatic download (~13GB,
budget time and Colab disk space for this) and sets up the train/val/test splits correctly.

In [ ]:
# This will trigger the dataset download on first run.
# If you want to sanity-check on a smaller slice first, see the "quick smoke test" cell below.
DATA_YAML = "SKU-110K.yaml"  # ships with ultralytics, no need to write our own (case-sensitive)

## Quick smoke test (optional but recommended first)

Before committing to a full multi-hour fine-tune, run a couple epochs on a small subset to
confirm the pipeline works end-to-end — catches config mistakes early instead of after a
2-hour training run fails at the last step.

In [ ]:
model = YOLO("yolov8n.pt")  # nano variant — fastest, good for a smoke test and for Colab's free GPU

results = model.train(
    data=DATA_YAML,
    epochs=2,
    imgsz=640,
    batch=16,
    project="grocery-detection",
    name="smoke-test",
)

## Full fine-tune

Once the smoke test runs clean, scale up. `yolov8n.pt` (nano) or `yolov8s.pt` (small) are the
right choices for a free Colab GPU — the larger variants (m/l/x) will be slow or run out of
memory. Adjust `epochs` based on how much Colab session time you have (free tier sessions can
disconnect after several hours — checkpoints save automatically so you can resume).

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    project="grocery-detection",
    name="sku110k-finetune",
    patience=10,       # stop early if validation performance plateaus
    save=True,
    resume=False,      # set True if resuming an interrupted run
)

## Evaluate

Standard detection metric is mAP (mean Average Precision) — how well predicted boxes match
ground truth across confidence thresholds. This is what goes in your README/interview talking
points as the actual number, not "it looked good."

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## Save the fine-tuned weights

Best weights are auto-saved during training at:
`grocery-detection/sku110k-finetune/weights/best.pt`

You'll need this file locally (or pushed to Hugging Face Hub — see Stage 12 in the tech stack
guide) so Phase 7 (pipeline integration) can load it without retraining.

- **On Kaggle**: download `best.pt` directly from the file browser panel in the left sidebar —
  no code needed.
- **On Colab**: use `google.colab.files.download("grocery-detection/sku110k-finetune/weights/best.pt")`
  if you need a programmatic download.

## Quick inference test

Run the fine-tuned model on a sample image to visually confirm it's detecting products
before moving on to Phase 4.

In [ ]:
# Replace with a path to any grocery product photo you have
test_results = model.predict("path/to/test_image.jpg", save=True, conf=0.25)
test_results[0].show()